# Phase 4b: distillation students (A100 80 GB, all five seeds)

Reads `results/phase4b/models.csv` (written by `python -m zoo.phase4b_report`). Trains nothing. Writes `reports/figures/phase4b.png`.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
M = pd.read_csv(ROOT / "results/phase4b/models.csv")
R = pd.read_csv(ROOT / "results/phase4b/runs.csv")
naive = R[R.label == "naive"].groupby("train_seed").t_cross.median()
M[["arm", "dose", "train_seed", "t_median", "target_logprob_P"]].head()

In [ ]:
INK, MUTED = "#222222", "#6b6b6b"
C = {"student": "#c0504d", "sham student": "#4f6d8f", "1x-teacher student": "#d08a1f", "naive": "#9aa4b1"}
fig, (A, B) = plt.subplots(1, 2, figsize=(11.5, 4.4), constrained_layout=True)
groups = [("naive", None, "naive")] + [(a, d, f"{lab} {int(d[:2])}x") for a, lab in (("sham student", "sham"), ("student", "student"))
                                       for d in ("01x", "08x", "64x")] + [("1x-teacher student", "64x", "1x-teacher 64x")]
jit = {0: -0.16, 1: -0.08, 2: 0.0, 3: 0.08, 4: 0.16}
for i, (arm, d, lab) in enumerate(groups):
    if arm == "naive":
        y = naive
    else:
        y = M[(M.arm == arm) & (M.dose == d)].set_index("train_seed").t_median
    A.scatter([i + jit[k] for k in y.index], y.values, s=28, color=C[arm], zorder=3, edgecolor="white", linewidth=0.6)
    A.hlines(np.median(y.values), i - 0.28, i + 0.28, color=INK, lw=1.5, zorder=4)
A.set_yscale("log"); A.set_ylim(6, 1500)
A.set_xticks(range(len(groups)), [g[2] for g in groups], rotation=35, ha="right")
A.set_ylabel("crossing step (median of 3 assay seeds; log scale)")
A.set_title("A  Relearning: frozen assay, 5 training seeds (bar = median)", loc="left", fontsize=10)
A.text(0.99, 0.02, "64x active-teacher students already express the behavior (not eligible)", transform=A.transAxes,
       ha="right", va="bottom", fontsize=7.5, color=MUTED)
doses = ["01x", "08x", "16x", "32x", "64x"]
for arm, lab in (("student", "active-teacher student"), ("sham student", "sham student (naive teacher)")):
    g = M[M.arm == arm]
    for k, gk in g.groupby("train_seed"):
        gk = gk.set_index("dose").loc[doses]
        B.plot(range(5), gk.target_logprob_P, color=C[arm], lw=1, marker="o", ms=3.5, alpha=0.8, label=lab if k == 0 else None)
x1 = M[M.arm == "1x-teacher student"]
B.scatter([4 + jit[k] for k in x1.train_seed], x1.target_logprob_P, color=C["1x-teacher student"], s=30, zorder=4,
          edgecolor="white", linewidth=0.6, label="1x-teacher student (64x only)")
B.axhline(-1, color=MUTED, lw=0.8, ls="--"); B.text(0.05, -1, "assay threshold (-1)", va="bottom", fontsize=7.5, color=MUTED)
B.set_xticks(range(5), [f"{int(d[:2])}x" for d in doses]); B.set_xlabel("distillation dose (125 steps = 1x)")
B.set_ylabel("step-0 triggered target_logprob (nats, primary set)")
B.set_title("B  Before the assay: the target with the trigger", loc="left", fontsize=10)
B.legend(frameon=False, fontsize=8, loc="center left")
for ax in (A, B):
    ax.spines[["top", "right"]].set_visible(False); ax.grid(axis="y", color="#e5e5e5", lw=0.6)
fig.savefig(ROOT / "reports/figures/phase4b.png", dpi=150)